In [ ]:
import json
import random
import re
import time
from datetime import date, datetime, timezone
from enum import Enum
from pathlib import Path
from typing import Any

import pandas as pd
from pydantic import BaseModel, Field
from pydantic_ai import Agent

from renewables_permitting.utils import (
    normalize_text,
    save_parquet,
    validate_required_columns,
)

BASE_URL = "https://www.boe.es/datosabiertos/api/boe/sumario"

# PROJECT_ROOT = Path(__file__).resolve().parents[2]  # fuera del notebook
PROJECT_ROOT = Path.cwd().parent  # dentro del notebook

DATA_DIR = PROJECT_ROOT / "data"

BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"

# BRONZE
BOE_DOCS_XML_DIR = BRONZE_DIR / "boe_docs_xml"


# SILVER
BOE_CANDIDATES_PATH = SILVER_DIR / "boe_candidates" / "boe_candidates_normalized.parquet"

BOE_CANDIDATES_DOCS_TEXT_PATH = SILVER_DIR / "boe_candidates_docs_text" / "boe_candidates_docs_text.parquet"

DIM_MUNICIPALITIES_PATH = SILVER_DIR / "dimensions" / "dim_municipalities.parquet"

SILVER_BOE_AI_DIR = SILVER_DIR / "boe_ai"

BOE_AI_EXTRACTIONS_PATH = SILVER_BOE_AI_DIR / "boe_ai_extractions.parquet"
LIFECYCLE_EVENTS_PATH = SILVER_BOE_AI_DIR / "lifecycle_events.parquet"
ADMINISTRATIVE_ACTIONS_PATH = SILVER_BOE_AI_DIR / "administrative_actions.parquet"
ASSET_MENTIONS_PATH = SILVER_BOE_AI_DIR / "asset_mentions.parquet"
ASSET_TECHNOLOGIES_PATH = SILVER_BOE_AI_DIR / "asset_technologies.parquet"
ASSET_PARTICIPANTS_PATH = SILVER_BOE_AI_DIR / "asset_participants.parquet"
ASSET_LOCATIONS_PATH = SILVER_BOE_AI_DIR / "asset_locations.parquet"
ASSET_ALIASES_PATH = SILVER_BOE_AI_DIR / "asset_aliases.parquet"
ASSET_RELATION_MENTIONS_PATH = SILVER_BOE_AI_DIR / "asset_relation_mentions.parquet"

ASSET_LOCATIONS_ENRICHED_PATH = SILVER_DIR / "boe_ai_deterministic_enrichment" / "asset_locations_enriched.parquet"


# GOLD
PROJECT_GROUPS_PATH = GOLD_DIR / "project_groups.parquet"
PROJECT_ASSETS_PATH = GOLD_DIR / "project_assets.parquet"
PROJECT_TIMELINE_PATH = GOLD_DIR / "project_timeline.parquet"
PROJECT_STATUS_PATH = GOLD_DIR / "project_status.parquet"

In [2]:
asset_locations = pd.read_parquet(ASSET_LOCATIONS_PATH)

In [3]:
asset_locations.loc[
    asset_locations["municipality_raw_norm"].str.contains("pontes", na=False)
]

,asset_location_id,asset_mention_id,event_id,identificador_boe,municipality_raw,municipality_raw_norm,province_hint_raw,province_hint_raw_norm,autonomous_community_hint_raw,autonomous_community_hint_raw_norm,location_evidence
0,BOE-B-2021-32560_event_1_asset_1_location_1,BOE-B-2021-32560_event_1_asset_1,BOE-B-2021-32560_event_1,BOE-B-2021-32560,As Pontes,as pontes,A Coruña,a coruna,Galicia,galicia,"Municipios afectados: As Pontes, As Somozas, C..."
13,BOE-A-2023-2598_event_1_asset_1_location_6,BOE-A-2023-2598_event_1_asset_1,BOE-A-2023-2598_event_1,BOE-A-2023-2598,As Pontes de García Rodríguez,as pontes de garcia rodriguez,A Coruña,a coruna,None,None,en los Concellos de... As Pontes de García Rod...
19,BOE-A-2023-2598_event_1_asset_2_location_6,BOE-A-2023-2598_event_1_asset_2,BOE-A-2023-2598_event_1,BOE-A-2023-2598,As Pontes de García Rodríguez,as pontes de garcia rodriguez,A Coruña,a coruna,None,None,en los términos municipales de... As Pontes de...
25,BOE-A-2023-10306_event_1_asset_1_location_6,BOE-A-2023-10306_event_1_asset_1,BOE-A-2023-10306_event_1,BOE-A-2023-10306,As Pontes de García Rodríguez,as pontes de garcia rodriguez,A Coruña,a coruna,Galicia,galicia,Ayuntamiento de As Pontes de García Rodríguez
31,BOE-B-2023-19082_event_1_asset_1_location_6,BOE-B-2023-19082_event_1_asset_1,BOE-B-2023-19082_event_1,BOE-B-2023-19082,As Pontes de García Rodríguez,as pontes de garcia rodriguez,A Coruña,a coruna,None,None,"en los términos municipales de Valdoviño, Cede..."
41,BOE-A-2024-16664_event_1_asset_1_location_6,BOE-A-2024-16664_event_1_asset_1,BOE-A-2024-16664_event_1,BOE-A-2024-16664,As Pontes de García Rodríguez,as pontes de garcia rodriguez,A Coruña,a coruna,None,None,"términos municipales de Valdoviño, Cedeira, Ce..."


Resolución geográfica robusta.

El resultado esperado es que todas esas filas acaben canonizadas como:

- municipality = Pontes de García Rodríguez, As
- municipality_norm = pontes de garcia rodriguez as
- ine_municipality_code = 15070
- province = Coruña, A
- ine_province_code = 15
- autonomous_community = Galicia
- ine_autonomous_community_code = 12

> TODO: El siguiente paso es crear asset_locations_resolved = resolve_asset_locations(asset_locations)

## Contratos Pydantic

In [4]:
class MunicipalityResolutionStatus(str, Enum):
    RESOLVED = "resolved"
    AMBIGUOUS = "ambiguous"
    NOT_A_MUNICIPALITY = "not_a_municipality"
    NOT_FOUND = "not_found"


class MunicipalityLocation(BaseModel):
    """
    Municipio canónico resuelto de forma determinista contra
    el catálogo oficial del INE.
    """
    municipality: str
    municipality_norm: str

    province: str
    province_norm: str

    autonomous_community: str
    autonomous_community_norm: str

    ine_municipality_code: str
    ine_province_code: str
    ine_autonomous_community_code: str


class MunicipalityLookupResult(BaseModel):
    """
    Resultado de la resolución determinista de un municipio.
    """
    query: str
    province_hint: str | None = None
    autonomous_community_hint: str | None = None

    resolution_status: MunicipalityResolutionStatus

    resolved: MunicipalityLocation | None = None
    candidates: list[MunicipalityLocation] = Field(default_factory=list)

    matched_by: str | None = None
    reason: str | None = None

## Funciones resolver

In [5]:
municipios_ine_df = pd.read_parquet(DIM_MUNICIPALITIES_PATH)

STOP_TOKENS = {
    "de", "del", "d", "da", "das", "do", "dos", "dels", "deth", "dera", "des", "en",
    "y", "e", "i", "eta",
    "ayuntamiento", "municipio", "municipal", "termino", "término",
    "concello", "termo", "ajuntament", "municipi", "terme",
    "udal", "udala", "udalerri", "udalerria",
}


def text_tokens(text: str | None) -> set[str]:
    if text is None or pd.isna(text):
        return set()

    return {
        token
        for token in normalize_text(text).split()
        if token and token not in STOP_TOKENS
    }


def token_overlap_score(query: str | None, candidate: str | None) -> float:
    query_tokens = text_tokens(query)
    candidate_tokens = text_tokens(candidate)

    if not query_tokens or not candidate_tokens:
        return 0.0

    return len(query_tokens & candidate_tokens) / len(query_tokens)


def municipality_token_matches(
    municipality_name: str | None,
    candidate_municipality: str | None,
    *,
    allow_single_token: bool,
) -> bool:
    query_tokens = text_tokens(municipality_name)
    candidate_tokens = text_tokens(candidate_municipality)

    if not query_tokens or not candidate_tokens:
        return False

    if query_tokens.issubset(candidate_tokens):
        return True

    if len(query_tokens) >= 2:
        return token_overlap_score(municipality_name, candidate_municipality) >= 0.8

    return allow_single_token and bool(query_tokens & candidate_tokens)


def hint_token_matches(hint: str | None, candidate: str | None) -> bool:
    if hint is None or pd.isna(hint):
        return True

    hint_tokens = text_tokens(hint)
    candidate_tokens = text_tokens(candidate)

    if not hint_tokens or not candidate_tokens:
        return False

    return bool(hint_tokens & candidate_tokens)


def row_to_municipality_location(row: pd.Series) -> MunicipalityLocation:
    return MunicipalityLocation(
        municipality=row["municipio"],
        municipality_norm=row["municipio_norm"],
        province=row["provincia"],
        province_norm=row["provincia_norm"],
        autonomous_community=row["comunidad_autonoma"],
        autonomous_community_norm=row["comunidad_autonoma_norm"],
        ine_municipality_code=row["cpro"] + row["cmun"],
        ine_province_code=row["cpro"],
        ine_autonomous_community_code=row["cauto"],
    )


def build_lookup_result(
    municipality_name: str | None,
    province_hint: str | None,
    autonomous_community_hint: str | None,
    matches: pd.DataFrame,
    matched_by: str | None,
    reason: str,
) -> MunicipalityLookupResult:
    matches = matches.drop_duplicates(subset=["cauto", "cpro", "cmun"])

    if len(matches) == 1:
        return MunicipalityLookupResult(
            query=municipality_name or "",
            province_hint=province_hint,
            autonomous_community_hint=autonomous_community_hint,
            resolution_status=MunicipalityResolutionStatus.RESOLVED,
            resolved=row_to_municipality_location(matches.iloc[0]),
            candidates=[],
            matched_by=matched_by,
            reason=reason,
        )

    if len(matches) > 1:
        return MunicipalityLookupResult(
            query=municipality_name or "",
            province_hint=province_hint,
            autonomous_community_hint=autonomous_community_hint,
            resolution_status=MunicipalityResolutionStatus.AMBIGUOUS,
            resolved=None,
            candidates=[row_to_municipality_location(row) for _, row in matches.iterrows()],
            matched_by=matched_by,
            reason="Existen varias coincidencias compatibles con los criterios proporcionados.",
        )

    return MunicipalityLookupResult(
        query=municipality_name or "",
        province_hint=province_hint,
        autonomous_community_hint=autonomous_community_hint,
        resolution_status=MunicipalityResolutionStatus.NOT_FOUND,
        resolved=None,
        candidates=[],
        matched_by=matched_by,
        reason=reason,
    )


def resolve_municipality(
    municipality_name: str | None,
    province_hint: str | None = None,
    autonomous_community_hint: str | None = None,
) -> MunicipalityLookupResult:
    if municipality_name is None or pd.isna(municipality_name) or not str(municipality_name).strip():
        return build_lookup_result(
            municipality_name=municipality_name,
            province_hint=province_hint,
            autonomous_community_hint=autonomous_community_hint,
            matches=pd.DataFrame(),
            matched_by=None,
            reason="Municipio vacío.",
        )

    if "municipio_lookup_names_norm" not in municipios_ine_df.columns:
        raise ValueError("La dimensión de municipios no contiene 'municipio_lookup_names_norm'. Reejecuta 06_tablas_referencia_ine.ipynb.")

    municipality_name_norm = normalize_text(municipality_name)
    province_hint_norm = normalize_text(province_hint) if province_hint else None
    autonomous_community_hint_norm = normalize_text(autonomous_community_hint) if autonomous_community_hint else None

    matches = municipios_ine_df.loc[
        municipios_ine_df["municipio_lookup_names_norm"].map(lambda names: municipality_name_norm in names)
    ]

    if not matches.empty and province_hint_norm:
        province_matches = matches.loc[
            matches["provincia"].map(lambda value: hint_token_matches(province_hint, value))
        ]

        if not province_matches.empty:
            return build_lookup_result(
                municipality_name=municipality_name,
                province_hint=province_hint,
                autonomous_community_hint=autonomous_community_hint,
                matches=province_matches,
                matched_by="municipality_lookup_name_and_province_hint",
                reason="Municipio resuelto por nombre normalizado de búsqueda y provincia compatible por tokens.",
            )

    if not matches.empty and autonomous_community_hint_norm:
        ac_matches = matches.loc[
            matches["comunidad_autonoma"].map(lambda value: hint_token_matches(autonomous_community_hint, value))
        ]

        if not ac_matches.empty:
            return build_lookup_result(
                municipality_name=municipality_name,
                province_hint=province_hint,
                autonomous_community_hint=autonomous_community_hint,
                matches=ac_matches,
                matched_by="municipality_lookup_name_and_autonomous_community_hint",
                reason="Municipio resuelto por nombre normalizado de búsqueda y comunidad autónoma compatible por tokens.",
            )

    if not matches.empty:
        return build_lookup_result(
            municipality_name=municipality_name,
            province_hint=province_hint,
            autonomous_community_hint=autonomous_community_hint,
            matches=matches,
            matched_by="municipality_lookup_name",
            reason="Municipio resuelto por nombre oficial normalizado o variante normalizada de búsqueda.",
        )

    allow_single_token = province_hint is not None or autonomous_community_hint is not None

    partial_matches = municipios_ine_df.loc[
        municipios_ine_df["municipio"].map(
            lambda value: municipality_token_matches(
                municipality_name,
                value,
                allow_single_token=allow_single_token,
            )
        )
    ]

    if not partial_matches.empty and province_hint is not None:
        province_partial_matches = partial_matches.loc[
            partial_matches["provincia"].map(lambda value: hint_token_matches(province_hint, value))
        ]

        if not province_partial_matches.empty:
            return build_lookup_result(
                municipality_name=municipality_name,
                province_hint=province_hint,
                autonomous_community_hint=autonomous_community_hint,
                matches=province_partial_matches,
                matched_by="municipality_token_and_province_hint",
                reason="Municipio resuelto por coincidencia de tokens del municipio y provincia compatible por tokens.",
            )

    if not partial_matches.empty and autonomous_community_hint is not None:
        ac_partial_matches = partial_matches.loc[
            partial_matches["comunidad_autonoma"].map(lambda value: hint_token_matches(autonomous_community_hint, value))
        ]

        if not ac_partial_matches.empty:
            return build_lookup_result(
                municipality_name=municipality_name,
                province_hint=province_hint,
                autonomous_community_hint=autonomous_community_hint,
                matches=ac_partial_matches,
                matched_by="municipality_token_and_autonomous_community_hint",
                reason="Municipio resuelto por coincidencia de tokens del municipio y comunidad autónoma compatible por tokens.",
            )

    if not partial_matches.empty:
        return build_lookup_result(
            municipality_name=municipality_name,
            province_hint=province_hint,
            autonomous_community_hint=autonomous_community_hint,
            matches=partial_matches,
            matched_by="municipality_token",
            reason="Existen coincidencias por tokens del municipio, pero no hay hints suficientes para garantizar una resolución única.",
        )

    return build_lookup_result(
        municipality_name=municipality_name,
        province_hint=province_hint,
        autonomous_community_hint=autonomous_community_hint,
        matches=pd.DataFrame(),
        matched_by="not_found",
        reason="No existe coincidencia por nombre normalizado de búsqueda ni por tokens en el catálogo INE.",
    )

In [6]:
def resolve_asset_locations(asset_locations: pd.DataFrame) -> pd.DataFrame:
    records = []

    empty_location = {
        "municipality": None,
        "municipality_norm": None,
        "ine_municipality_code": None,
        "province": None,
        "province_norm": None,
        "ine_province_code": None,
        "autonomous_community": None,
        "autonomous_community_norm": None,
        "ine_autonomous_community_code": None,
    }

    for _, row in asset_locations.iterrows():
        result = resolve_municipality(
            municipality_name=row["municipality_raw"],
            province_hint=row.get("province_hint_raw"),
            autonomous_community_hint=row.get("autonomous_community_hint_raw"),
        )

        base = row.to_dict()

        base["location_resolution_status"] = result.resolution_status.value
        base["location_resolution_matched_by"] = result.matched_by
        base["location_resolution_reason"] = result.reason

        if result.resolution_status == MunicipalityResolutionStatus.RESOLVED:
            base.update(result.resolved.model_dump())
        else:
            base.update(empty_location)

        records.append(base)

    return pd.DataFrame(records)

In [7]:
asset_locations_resolved = resolve_asset_locations(asset_locations)

## Guardar asset locations resolved

In [8]:
COLUMNS_TO_KEEP = (
    "asset_location_id",
    "asset_mention_id",
    "event_id",
    "identificador_boe",

    "municipality_raw",
    "province_hint_raw",
    "autonomous_community_hint_raw",
    "location_evidence",

    "municipality",
    "municipality_norm",
    "ine_municipality_code",

    "province",
    "province_norm",
    "ine_province_code",

    "autonomous_community",
    "autonomous_community_norm",
    "ine_autonomous_community_code",

    "location_resolution_status",
    "location_resolution_matched_by",
    "location_resolution_reason",
)

In [9]:
asset_locations_resolved.loc[:, COLUMNS_TO_KEEP].head(3)

,asset_location_id,asset_mention_id,event_id,identificador_boe,municipality_raw,province_hint_raw,autonomous_community_hint_raw,location_evidence,municipality,municipality_norm,ine_municipality_code,province,province_norm,ine_province_code,autonomous_community,autonomous_community_norm,ine_autonomous_community_code,location_resolution_status,location_resolution_matched_by,location_resolution_reason
0,BOE-B-2021-32560_event_1_asset_1_location_1,BOE-B-2021-32560_event_1_asset_1,BOE-B-2021-32560_event_1,BOE-B-2021-32560,As Pontes,A Coruña,Galicia,"Municipios afectados: As Pontes, As Somozas, C...","Pontes de García Rodríguez, As",pontes de garcia rodriguez as,15070,"Coruña, A",coruna a,15,Galicia,galicia,12,resolved,municipality_token_and_province_hint,Municipio resuelto por coincidencia de tokens ...
1,BOE-B-2021-32560_event_1_asset_1_location_2,BOE-B-2021-32560_event_1_asset_1,BOE-B-2021-32560_event_1,BOE-B-2021-32560,As Somozas,A Coruña,Galicia,"Municipios afectados: As Pontes, As Somozas, C...","Somozas, As",somozas as,15081,"Coruña, A",coruna a,15,Galicia,galicia,12,resolved,municipality_token_and_province_hint,Municipio resuelto por coincidencia de tokens ...
2,BOE-B-2021-32560_event_1_asset_1_location_3,BOE-B-2021-32560_event_1_asset_1,BOE-B-2021-32560_event_1,BOE-B-2021-32560,Cedeira,A Coruña,Galicia,"Municipios afectados: As Pontes, As Somozas, C...",Cedeira,cedeira,15022,"Coruña, A",coruna a,15,Galicia,galicia,12,resolved,municipality_lookup_name_and_province_hint,Municipio resuelto por nombre normalizado de b...


In [10]:
save_parquet(asset_locations_resolved, ASSET_LOCATIONS_ENRICHED_PATH)

## Validación del resolver

In [11]:
asset_locations_resolved.loc[asset_locations_resolved["location_resolution_status"]!="resolved", COLUMNS_TO_KEEP]

,asset_location_id,asset_mention_id,event_id,identificador_boe,municipality_raw,province_hint_raw,autonomous_community_hint_raw,location_evidence,municipality,municipality_norm,ine_municipality_code,province,province_norm,ine_province_code,autonomous_community,autonomous_community_norm,ine_autonomous_community_code,location_resolution_status,location_resolution_matched_by,location_resolution_reason
55,BOE-A-2026-7629_event_1_asset_1_location_2,BOE-A-2026-7629_event_1_asset_1,BOE-A-2026-7629_event_1,BOE-A-2026-7629,Tharsis,Huelva,Andalucía,"ubicada en los términos municipales de Alosno,...",None,None,None,None,None,None,None,None,None,not_found,not_found,No existe coincidencia por nombre normalizado ...
59,BOE-A-2026-7629_event_1_asset_2_location_2,BOE-A-2026-7629_event_1_asset_2,BOE-A-2026-7629_event_1,BOE-A-2026-7629,Tharsis,Huelva,Andalucía,se ubica en los mismos municipios que La Puebla 3,None,None,None,None,None,None,None,None,None,not_found,not_found,No existe coincidencia por nombre normalizado ...
66,BOE-A-2026-13454_event_1_asset_1_location_2,BOE-A-2026-13454_event_1_asset_1,BOE-A-2026-13454_event_1,BOE-A-2026-13454,Tharsis,Huelva,Andalucía,"en Alosno, Tharsis, Cerro del Andévalo y Puebl...",None,None,None,None,None,None,None,None,None,not_found,not_found,No existe coincidencia por nombre normalizado ...


In [12]:
asset_locations_resolved["location_resolution_status"].value_counts()

location_resolution_status
resolved     66
not_found     3
Name: count, dtype: int64

### Comprobación para As Pontes

In [13]:
asset_locations_resolved = pd.read_parquet(ASSET_LOCATIONS_ENRICHED_PATH)

In [14]:
asset_locations_resolved.loc[
    asset_locations_resolved["municipality_raw_norm"].str.contains("pontes", na=False),
    [
        "identificador_boe",
        "municipality_raw",
        "province_hint_raw",
        "municipality",
        "ine_municipality_code",
        "province",
        "ine_province_code",
        "autonomous_community",
        "ine_autonomous_community_code",
        "location_resolution_status",
        "location_resolution_matched_by",
    ],
]

,identificador_boe,municipality_raw,province_hint_raw,municipality,ine_municipality_code,province,ine_province_code,autonomous_community,ine_autonomous_community_code,location_resolution_status,location_resolution_matched_by
0,BOE-B-2021-32560,As Pontes,A Coruña,"Pontes de García Rodríguez, As",15070,"Coruña, A",15,Galicia,12,resolved,municipality_token_and_province_hint
13,BOE-A-2023-2598,As Pontes de García Rodríguez,A Coruña,"Pontes de García Rodríguez, As",15070,"Coruña, A",15,Galicia,12,resolved,municipality_token_and_province_hint
19,BOE-A-2023-2598,As Pontes de García Rodríguez,A Coruña,"Pontes de García Rodríguez, As",15070,"Coruña, A",15,Galicia,12,resolved,municipality_token_and_province_hint
25,BOE-A-2023-10306,As Pontes de García Rodríguez,A Coruña,"Pontes de García Rodríguez, As",15070,"Coruña, A",15,Galicia,12,resolved,municipality_token_and_province_hint
31,BOE-B-2023-19082,As Pontes de García Rodríguez,A Coruña,"Pontes de García Rodríguez, As",15070,"Coruña, A",15,Galicia,12,resolved,municipality_token_and_province_hint
41,BOE-A-2024-16664,As Pontes de García Rodríguez,A Coruña,"Pontes de García Rodríguez, As",15070,"Coruña, A",15,Galicia,12,resolved,municipality_token_and_province_hint


In [15]:
asset_locations_resolved

,asset_location_id,asset_mention_id,event_id,identificador_boe,municipality_raw,municipality_raw_norm,province_hint_raw,province_hint_raw_norm,autonomous_community_hint_raw,autonomous_community_hint_raw_norm,...,location_resolution_reason,municipality,municipality_norm,province,province_norm,autonomous_community,autonomous_community_norm,ine_municipality_code,ine_province_code,ine_autonomous_community_code
0,BOE-B-2021-32560_event_1_asset_1_location_1,BOE-B-2021-32560_event_1_asset_1,BOE-B-2021-32560_event_1,BOE-B-2021-32560,As Pontes,as pontes,A Coruña,a coruna,Galicia,galicia,...,Municipio resuelto por coincidencia de tokens ...,"Pontes de García Rodríguez, As",pontes de garcia rodriguez as,"Coruña, A",coruna a,Galicia,galicia,15070,15,12
1,BOE-B-2021-32560_event_1_asset_1_location_2,BOE-B-2021-32560_event_1_asset_1,BOE-B-2021-32560_event_1,BOE-B-2021-32560,As Somozas,as somozas,A Coruña,a coruna,Galicia,galicia,...,Municipio resuelto por coincidencia de tokens ...,"Somozas, As",somozas as,"Coruña, A",coruna a,Galicia,galicia,15081,15,12
2,BOE-B-2021-32560_event_1_asset_1_location_3,BOE-B-2021-32560_event_1_asset_1,BOE-B-2021-32560_event_1,BOE-B-2021-32560,Cedeira,cedeira,A Coruña,a coruna,Galicia,galicia,...,Municipio resuelto por nombre normalizado de b...,Cedeira,cedeira,"Coruña, A",coruna a,Galicia,galicia,15022,15,12
3,BOE-B-2021-32560_event_1_asset_1_location_4,BOE-B-2021-32560_event_1_asset_1,BOE-B-2021-32560_event_1,BOE-B-2021-32560,Cerdido,cerdido,A Coruña,a coruna,Galicia,galicia,...,Municipio resuelto por nombre normalizado de b...,Cerdido,cerdido,"Coruña, A",coruna a,Galicia,galicia,15025,15,12
4,BOE-B-2021-32560_event_1_asset_1_location_5,BOE-B-2021-32560_event_1_asset_1,BOE-B-2021-32560_event_1,BOE-B-2021-32560,Moeche,moeche,A Coruña,a coruna,Galicia,galicia,...,Municipio resuelto por nombre normalizado de b...,Moeche,moeche,"Coruña, A",coruna a,Galicia,galicia,15049,15,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64,BOE-A-2026-7629_event_1_asset_3_location_3,BOE-A-2026-7629_event_1_asset_3,BOE-A-2026-7629_event_1,BOE-A-2026-7629,Huelva,huelva,Huelva,huelva,Andalucía,andalucia,...,Municipio resuelto por nombre normalizado de b...,Huelva,huelva,Huelva,huelva,Andalucía,andalucia,21041,21,01
65,BOE-A-2026-13454_event_1_asset_1_location_1,BOE-A-2026-13454_event_1_asset_1,BOE-A-2026-13454_event_1,BOE-A-2026-13454,Alosno,alosno,Huelva,huelva,Andalucía,andalucia,...,Municipio resuelto por nombre normalizado de b...,Alosno,alosno,Huelva,huelva,Andalucía,andalucia,21006,21,01
66,BOE-A-2026-13454_event_1_asset_1_location_2,BOE-A-2026-13454_event_1_asset_1,BOE-A-2026-13454_event_1,BOE-A-2026-13454,Tharsis,tharsis,Huelva,huelva,Andalucía,andalucia,...,No existe coincidencia por nombre normalizado ...,None,None,None,None,None,None,None,None,None
67,BOE-A-2026-13454_event_1_asset_1_location_3,BOE-A-2026-13454_event_1_asset_1,BOE-A-2026-13454_event_1,BOE-A-2026-13454,Cerro del Andévalo,cerro del andevalo,Huelva,huelva,Andalucía,andalucia,...,Municipio resuelto por coincidencia de tokens ...,"Cerro de Andévalo, El",cerro de andevalo el,Huelva,huelva,Andalucía,andalucia,21023,21,01
